# Xgboost + SMOTE

In [ ]:
# Imports principais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_csv("..\\datasets_paralelos\\kmeans_5clusters.csv")

In [ ]:
# 2. Separar X e y
X = df.drop(columns=['cluster'])
Y = df['cluster']

In [ ]:
# 3. Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.3, random_state=42, stratify=Y
)

In [ ]:
model = xgb_model = XGBClassifier(
    objective='multi:softprob',   # para multiclasse com probabilidades
    num_class=5,                  # número de classes (clusters)
    eval_metric='mlogloss',       # boa métrica para multiclasse
    use_label_encoder=False,      # evita warning em versões novas

    # Hiperparâmetros iniciais
    n_estimators=200,             # número de árvores (boosting rounds)
    max_depth=6,                  # profundidade máxima de cada árvore
    learning_rate=0.1,            # taxa de aprendizado
    subsample=0.8,                # fração de amostras para cada árvore
    colsample_bytree=0.8,         # fração de features por árvore
    gamma=0,                      # regularização para split
    reg_alpha=0.1,                # L1 regularization
    reg_lambda=1,                 # L2 regularization

    random_state=42,
    verbosity=1
)

In [ ]:
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),  # SMOTE para balanceamento
    ('clf', model)]) # Classificador XGBoost     

In [ ]:
# 5. Cross-validation
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print("Acurácias de cross-validation:", cv_scores)
print("Média da acurácia de cross-validation:", np.mean(cv_scores))

In [ ]:
# 6. Treinar o pipeline completo no treino
pipeline.fit(X_train, y_train)

In [ ]:
# 7. Avaliar no treino
y_train_pred = pipeline.predict(X_train)
print('Métricas de Classificação para o conjunto de treino:')
print(classification_report(y_train, y_train_pred))

In [ ]:
# 8. Avaliar no teste
y_test_pred = pipeline.predict(X_test)
print("Métricas de Classificação para o conjunto de teste:")
print(classification_report(y_test, y_test_pred))

In [ ]:
# 9. Matriz de confusão do teste
conf_matrix = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(8,6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=sorted(Y.unique()), yticklabels=sorted(Y.unique()))
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão - Teste')
plt.show()

In [ ]:

# Matriz de Confusão para o conjunto de treino
conf_matrix = confusion_matrix(y_train, y_train_pred)
plt.figure(figsize=(8,6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=sorted(Y.unique()), yticklabels=sorted(Y.unique()))
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão - Treino')
plt.show()

